# 🏙️ Paris Housing — Notebook 02: Feature Engineering & Preprocessing

Prepare the dataset for machine learning models.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

sns.set_theme(style="whitegrid")
SEED = 42
print("Libraries loaded")


## 1. Load Raw Data


In [ ]:
df = pd.read_csv("../data/paris_housing_prices_dataset.csv")
print(df.shape)
df.head(3)


## 2. New Features


In [ ]:
# Price per square metre (useful EDA metric — not used as input feature)
df["price_per_sqm"] = df["Price_EUR"] / df["Size_sqm"]

# Property age at time of sale (assume data is from 2024)
df["property_age"] = 2024 - df["Year_Built"]

# Is the property in a central arrondissement (1-8)?
df["is_central"] = (df["Arrondissement"] <= 8).astype(int)

# Room density (size per room)
df["sqm_per_room"] = df["Size_sqm"] / df["Rooms"]

print("New features added:", ["price_per_sqm","property_age","is_central","sqm_per_room"])
df[["price_per_sqm","property_age","is_central","sqm_per_room"]].describe().round(2)


## 3. Encode Categorical Variables


In [ ]:
le_type = LabelEncoder()
le_cond = LabelEncoder()

df["Property_Type_enc"] = le_type.fit_transform(df["Property_Type"])
df["Condition_enc"]     = le_cond.fit_transform(df["Condition"])

print("Property_Type mapping:", dict(zip(le_type.classes_, le_type.transform(le_type.classes_))))
print("Condition mapping    :", dict(zip(le_cond.classes_, le_cond.transform(le_cond.classes_))))


## 4. Define Features & Target — Train / Test Split


In [ ]:
FEATURES = [
    "Arrondissement", "Size_sqm", "Rooms", "Floor",
    "property_age", "Distance_to_Center_km",
    "is_central", "sqm_per_room",
    "Property_Type_enc", "Condition_enc"
]
TARGET = "Price_EUR"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")


## 5. Scale Features


In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Scaling complete. Mean of first feature (train):", X_train_sc[:,0].mean().round(6))


## 6. Save Processed Arrays


In [ ]:
np.save("../outputs/X_train_sc.npy",  X_train_sc)
np.save("../outputs/X_test_sc.npy",   X_test_sc)
np.save("../outputs/y_train.npy",     y_train.values)
np.save("../outputs/y_test.npy",      y_test.values)

# Also save raw (unscaled) splits for tree models
np.save("../outputs/X_train_raw.npy", X_train.values)
np.save("../outputs/X_test_raw.npy",  X_test.values)

print("Saved all arrays to ../outputs/")
print("Feature list:", FEATURES)


**→ Proceed to Notebook 03: Modelling**
